# 02 — LangGraph Basics
**Study Notebook 2 of 3 · Estimated time: 30 minutes**

By the end of this notebook you will:
- Build a state graph with typed state, nodes, and edges
- Add conditional routing (branching)
- Add a human-in-the-loop approval gate
- Visualise the graph structure

> **Prereq:** No API key needed for this notebook — LangGraph is pure Python orchestration.

---
## Step 0 — Install

In [ ]:
import sys
!{sys.executable} -m pip install -q langgraph

---
## Step 1 — Minimal State Graph

A LangGraph graph has three ingredients:
1. **State** — a `TypedDict` that travels through every node
2. **Nodes** — Python functions that read and update the state
3. **Edges** — connections that define what runs next

From **slide 6**: LangGraph uses *directed graph architecture* instead of raw LLM decisions.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

# 1. Define the state shape
class PipelineState(TypedDict):
    input_text: str
    processed_text: str
    word_count: int

# 2. Define nodes (functions that update state)
def preprocess_node(state: PipelineState) -> dict:
    """Clean and normalise the input text."""
    cleaned = state["input_text"].strip().lower()
    print(f"[preprocess] '{state['input_text']}' → '{cleaned}'")
    return {"processed_text": cleaned}

def count_node(state: PipelineState) -> dict:
    """Count words in the processed text."""
    count = len(state["processed_text"].split())
    print(f"[count] {count} words found")
    return {"word_count": count}

# 3. Build the graph
builder = StateGraph(PipelineState)
builder.add_node("preprocess", preprocess_node)
builder.add_node("count", count_node)

builder.set_entry_point("preprocess")
builder.add_edge("preprocess", "count")
builder.add_edge("count", END)

graph = builder.compile()

# Run it
result = graph.invoke({
    "input_text": "  FOSS is Freedom  ",
    "processed_text": "",
    "word_count": 0
})
print(f"\nFinal state: {result}")

**Notice:** The state `TypedDict` acts as the single *auditable source of truth* (slide 6). Each node only updates the keys it cares about — the rest flow through untouched.

---
## Step 2 — Conditional Routing

Real pipelines need branching. LangGraph uses **router functions** that return the name of the next node.

In [ ]:
from typing import Literal

class ReviewState(TypedDict):
    text: str
    word_count: int
    verdict: str

def analyse_node(state: ReviewState) -> dict:
    count = len(state["text"].split())
    print(f"[analyse] {count} words")
    return {"word_count": count}

def approve_node(state: ReviewState) -> dict:
    print("[approve] Content approved — long enough.")
    return {"verdict": "approved"}

def reject_node(state: ReviewState) -> dict:
    print("[reject] Content too short — rejected.")
    return {"verdict": "rejected"}

# Router function — returns the next node name as a string
def route_by_length(state: ReviewState) -> Literal["approve", "reject"]:
    if state["word_count"] >= 5:
        return "approve"
    return "reject"

builder2 = StateGraph(ReviewState)
builder2.add_node("analyse", analyse_node)
builder2.add_node("approve", approve_node)
builder2.add_node("reject", reject_node)

builder2.set_entry_point("analyse")
builder2.add_conditional_edges("analyse", route_by_length)
builder2.add_edge("approve", END)
builder2.add_edge("reject", END)

graph2 = builder2.compile()

# Test with short text
print("--- Short text ---")
graph2.invoke({"text": "Hello world", "word_count": 0, "verdict": ""})

print()
print("--- Long text ---")
graph2.invoke({"text": "FOSS is freedom and the future of software", "word_count": 0, "verdict": ""})

---
## Step 3 — Human-in-the-Loop Gate

From **slide 6**: LangGraph can pause at any node and wait for human input before continuing. This is the *human gate* from the slides.

We use `interrupt_before` + a `MemorySaver` checkpointer to persist state between interrupts.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

class ApprovalState(TypedDict):
    proposal: str
    approved: bool
    outcome: str

def draft_node(state: ApprovalState) -> dict:
    print(f"[draft] Proposal ready: '{state['proposal']}'")
    print(">>> PAUSED — waiting for human approval <<<")
    return {}

def execute_node(state: ApprovalState) -> dict:
    if state["approved"]:
        print("[execute] Human approved — running proposal.")
        return {"outcome": "executed"}
    else:
        print("[execute] Human rejected — aborting.")
        return {"outcome": "aborted"}

checkpointer = MemorySaver()
builder3 = StateGraph(ApprovalState)
builder3.add_node("draft", draft_node)
builder3.add_node("execute", execute_node)
builder3.set_entry_point("draft")
builder3.add_edge("draft", "execute")
builder3.add_edge("execute", END)

# interrupt_before tells LangGraph to pause BEFORE running "execute"
gate_graph = builder3.compile(
    checkpointer=checkpointer,
    interrupt_before=["execute"]
)

config = {"configurable": {"thread_id": "demo-1"}}
initial_state = {"proposal": "Deploy v2.0 to production", "approved": False, "outcome": ""}

# First run — stops at the human gate
print("=== First run (stops at gate) ===")
for event in gate_graph.stream(initial_state, config):
    pass

print()
print("=== Human approves — resume ===")
# Resume with human decision injected into state
gate_graph.update_state(config, {"approved": True})
for event in gate_graph.stream(None, config):
    pass

---
## Step 4 — Visualise the Graph

> Requires `graphviz` and `pillow` in Colab — the install cell covers this.

In [ ]:
!pip install -q grandalf

In [ ]:
# ASCII graph (works everywhere without graphviz)
print(graph2.get_graph().draw_ascii())

In [ ]:
# Mermaid diagram (paste into mermaid.live to visualise)
print(graph2.get_graph().draw_mermaid())

---
## Exercise — Add a Third Node

Extend `graph2` to add a `flag_for_review` node. Route to it when word count is between 3 and 4 (neither clearly short nor clearly long).

In [ ]:
# Your solution here
# Hint: update route_by_length to return a third option: "flag_for_review"
# and add a new node + edge for it

def flag_for_review_node(state):
    print("[flag] Borderline content — flagged for manual review.")
    return {"verdict": "flagged"}

# TODO: wire it into the graph

---
## Summary

| Concept | LangGraph tool |
|---------|----------------|
| Typed state dict | `TypedDict` |
| Graph builder | `StateGraph` |
| Linear flow | `add_edge(A, B)` |
| Conditional routing | `add_conditional_edges(node, router_fn)` |
| Human gate (pause) | `interrupt_before=["node_name"]` + `MemorySaver` |
| Resume after gate | `update_state()` then `stream(None, config)` |

**Next:** `03_a2a_and_hybrid.ipynb` — learn the A2A protocol and wire 3 specialist agents into LangGraph to build the FOSS Contribution Matchmaker.